In [ ]:
!pip -q install transformers torch sentence-transformers faiss-cpu pandas numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 36.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

from transformers import pipeline
from sentence_transformers import SentenceTransformer
import faiss


In [ ]:
GENRES = [
    "Fantasy", "Romance", "Mystery", "Science Fiction", "Self-Help",
    "History", "Business", "Children", "Horror", "Poetry"
]

zero_shot = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli"
)

def classify_genre(text: str):
    out = zero_shot(text, candidate_labels=GENRES, multi_label=False)
    return out["labels"][0], float(out["scores"][0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [ ]:
np.random.seed(42)

templates = {
    "Fantasy": [
        "A young hero discovers a hidden kingdom and must defeat a dark sorcerer.",
        "Dragons rise again as an ancient prophecy awakens in the north."
    ],
    "Romance": [
        "Two strangers meet in a small cafe and find love against all odds.",
        "A long-distance relationship is tested by secrets and time."
    ],
    "Mystery": [
        "A detective investigates a series of murders in a quiet town.",
        "A missing diary reveals clues to an old family crime."
    ],
    "Science Fiction": [
        "A crew travels through a wormhole to save humanity from collapse.",
        "An AI gains consciousness and changes the future of Earth."
    ],
    "Self-Help": [
        "A practical guide to build habits and improve focus every day.",
        "Learn to manage anxiety with simple routines and mindset shifts."
    ],
    "History": [
        "An account of ancient empires and the wars that shaped the world.",
        "A deep dive into the political revolutions of the 20th century."
    ],
    "Business": [
        "How startups scale products and build strong teams.",
        "Negotiation tactics and leadership strategies for managers."
    ],
    "Children": [
        "A curious cat explores the city and learns about friendship.",
        "A magical school adventure for kids with puzzles and fun."
    ],
    "Horror": [
        "A haunted house whispers at night, luring visitors inside.",
        "A village faces a terrifying creature in the woods."
    ],
    "Poetry": [
        "A collection of poems about love, loss, and hope.",
        "Minimalist poems inspired by nature and silence."
    ]
}

def make_books(n_books=200):
    rows = []
    for i in range(1, n_books + 1):
        g = np.random.choice(GENRES)
        desc = np.random.choice(templates[g])
        title = f"{g} Book {i}"
        author = f"Author {np.random.randint(1, 60)}"
        rows.append([i, title, author, desc])
    return pd.DataFrame(rows, columns=["book_id", "title", "author", "description"])

books_df = make_books(200)
books_df.head()


,book_id,title,author,description
0,1,Business Book 1,Author 29,Negotiation tactics and leadership strategies ...
1,2,Children Book 2,Author 21,A curious cat explores the city and learns abo...
2,3,Business Book 3,Author 19,Negotiation tactics and leadership strategies ...
3,4,Business Book 4,Author 11,How startups scale products and build strong t...
4,5,Children Book 5,Author 36,A curious cat explores the city and learns abo...


In [ ]:
genres = []
scores = []

for txt in books_df["title"] + " | " + books_df["description"]:
    g, s = classify_genre(txt)
    genres.append(g)
    scores.append(s)

books_df["genre"] = genres
books_df["genre_confidence"] = scores

books_df[["book_id", "title", "genre", "genre_confidence"]].head(10)


,book_id,title,genre,genre_confidence
0,1,Business Book 1,Business,0.959846
1,2,Children Book 2,Children,0.785824
2,3,Business Book 3,Business,0.940125
3,4,Business Book 4,Business,0.919759
4,5,Children Book 5,Children,0.736310
5,6,Children Book 6,Children,0.777509
6,7,History Book 7,History,0.966959
7,8,Children Book 8,Children,0.798004
8,9,History Book 9,History,0.953918
9,10,Fantasy Book 10,Fantasy,0.534217


In [ ]:
embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

books_df["text"] = (
    "Title: " + books_df["title"] +
    " | Author: " + books_df["author"] +
    " | Genre: " + books_df["genre"] +
    " | Description: " + books_df["description"]
)

book_embeddings = embed_model.encode(
    books_df["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

dim = book_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)      # cosine similarity (لأننا normalized)
index.add(book_embeddings)

print("Embeddings:", book_embeddings.shape)
print("FAISS size:", index.ntotal)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embeddings: (200, 384)
FAISS size: 200


In [ ]:
def make_interactions(n_users=50, views_per_user=30, likes_per_user=8):
    events = []
    now = datetime.now()

    for u in range(1, n_users + 1):
        viewed = np.random.choice(books_df["book_id"], size=views_per_user, replace=False)
        liked = np.random.choice(viewed, size=likes_per_user, replace=False)

        for b in viewed:
            ts = now - timedelta(days=int(np.random.randint(0, 30)))
            events.append([u, int(b), "view", ts.isoformat()])

        for b in liked:
            ts = now - timedelta(days=int(np.random.randint(0, 30)))
            events.append([u, int(b), "like", ts.isoformat()])

    return pd.DataFrame(events, columns=["user_id", "book_id", "event_type", "timestamp"])

interactions_df = make_interactions()
interactions_df.head(10)


,user_id,book_id,event_type,timestamp
0,1,144,view,2026-02-21T17:14:45.542474
1,1,119,view,2026-03-11T17:14:45.542474
2,1,107,view,2026-02-23T17:14:45.542474
3,1,43,view,2026-03-11T17:14:45.542474
4,1,184,view,2026-03-06T17:14:45.542474
5,1,118,view,2026-03-06T17:14:45.542474
6,1,55,view,2026-02-25T17:14:45.542474
7,1,131,view,2026-02-27T17:14:45.542474
8,1,195,view,2026-02-23T17:14:45.542474
9,1,48,view,2026-03-02T17:14:45.542474


In [ ]:
EVENT_W = {"view": 1.0, "like": 3.0}
bookid_to_idx = {bid: i for i, bid in enumerate(books_df["book_id"].tolist())}

def user_interest_ratios(user_id, interactions_df):
    u = interactions_df[interactions_df.user_id == user_id].merge(
        books_df[["book_id", "genre"]], on="book_id", how="left"
    )
    if u.empty:
        return {g: 1/len(GENRES) for g in GENRES}

    u["w"] = u["event_type"].map(EVENT_W).fillna(0.0)
    s = u.groupby("genre")["w"].sum().reindex(GENRES, fill_value=0.0)

    total = s.sum()
    if total == 0:
        return {g: 1/len(GENRES) for g in GENRES}

    return (s / total).to_dict()

def build_user_vector(user_id, interactions_df):
    u = interactions_df[interactions_df.user_id == user_id]
    if u.empty:
        return None, set()

    vecs, weights, seen = [], [], set()
    for _, row in u.iterrows():
        bid = int(row["book_id"])
        ev  = row["event_type"]
        if bid not in bookid_to_idx:
            continue
        w = EVENT_W.get(ev, 0.0)
        if w == 0:
            continue
        vecs.append(book_embeddings[bookid_to_idx[bid]])
        weights.append(w)
        seen.add(bid)

    if not vecs:
        return None, seen

    vecs = np.array(vecs)
    weights = np.array(weights).reshape(-1, 1)

    user_vec = np.sum(vecs * weights, axis=0) / (np.sum(np.abs(weights)) + 1e-9)
    user_vec = user_vec / (np.linalg.norm(user_vec) + 1e-9)
    return user_vec.astype("float32"), seen


In [ ]:
uid = 1

print("Interest ratios:", user_interest_ratios(uid, interactions_df))

feed = build_mixed_feed(uid, interactions_df, feed_size=50)
feed.head(10)


Interest ratios: {'Fantasy': 0.1111111111111111, 'Romance': 0.07407407407407407, 'Mystery': 0.2037037037037037, 'Science Fiction': 0.2222222222222222, 'Self-Help': 0.05555555555555555, 'History': 0.05555555555555555, 'Business': 0.05555555555555555, 'Children': 0.1111111111111111, 'Horror': 0.0, 'Poetry': 0.1111111111111111}


,book_id,title,author,genre,score
0,109,Mystery Book 109,Author 24,Mystery,0.799200
1,169,Mystery Book 169,Author 58,Mystery,0.791255
2,93,Mystery Book 93,Author 59,Mystery,0.789383
3,37,Fantasy Book 37,Author 7,Fantasy,0.768045
4,151,Science Fiction Book 151,Author 20,Science Fiction,0.758007
5,79,Fantasy Book 79,Author 49,Fantasy,0.757395
6,96,Science Fiction Book 96,Author 7,Science Fiction,0.753472
7,97,Science Fiction Book 97,Author 33,Science Fiction,0.752389
8,134,Horror Book 134,Author 24,Mystery,0.750338
9,166,Science Fiction Book 166,Author 13,Science Fiction,0.749133


In [ ]:
def allocate_feed(ratios: dict, unseen_counts: dict, feed_size=50):
    alloc = {g: 0 for g in GENRES}
    remaining = feed_size

    # target counts proportional to ratios
    target = {g: int(round(ratios.get(g, 0.0) * feed_size)) for g in GENRES}

    # cap by availability
    for g in GENRES:
        alloc[g] = min(target[g], unseen_counts.get(g, 0))
        remaining -= alloc[g]

    # distribute leftovers to best ratio genres that still have items
    while remaining > 0:
        candidates = [g for g in GENRES if alloc[g] < unseen_counts.get(g, 0)]
        if not candidates:
            break
        g = max(candidates, key=lambda x: ratios.get(x, 0.0))
        alloc[g] += 1
        remaining -= 1

    return alloc


In [ ]:
def build_mixed_feed(user_id, interactions_df, feed_size=50, random_state=42):
    ratios = user_interest_ratios(user_id, interactions_df)

    seen = set(interactions_df.loc[interactions_df.user_id == user_id, "book_id"].astype(int).tolist())
    unseen_df = books_df[~books_df.book_id.isin(seen)].copy()

    unseen_counts = unseen_df.groupby("genre")["book_id"].count().reindex(GENRES, fill_value=0).to_dict()
    alloc = allocate_feed(ratios, unseen_counts, feed_size=feed_size)

    parts = []
    for g, k in alloc.items():
        if k <= 0:
            continue
        g_df = unseen_df[unseen_df.genre == g]
        if len(g_df) == 0:
            continue
        parts.append(g_df.sample(n=min(k, len(g_df)), random_state=random_state))

    if not parts:
        return books_df.sample(feed_size, random_state=random_state)[["book_id","title","author","genre"]]

    feed = pd.concat(parts, ignore_index=True)

    # Shuffle / ranking
    feed = feed.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

    user_vec, _ = build_user_vector(user_id, interactions_df)
    if user_vec is not None:
        idxs = [bookid_to_idx[int(b)] for b in feed["book_id"].tolist()]
        feed_vecs = book_embeddings[idxs]
        feed["score"] = (feed_vecs @ user_vec).astype(float)
        feed = feed.sort_values("score", ascending=False).reset_index(drop=True)

    cols = ["book_id","title","author","genre"]
    if "score" in feed.columns:
        cols.append("score")
    return feed[cols]


In [ ]:
uid = 1
feed = build_mixed_feed(uid, interactions_df, feed_size=50)
feed.head(15)


,book_id,title,author,genre,score
0,109,Mystery Book 109,Author 24,Mystery,0.799200
1,169,Mystery Book 169,Author 58,Mystery,0.791255
2,93,Mystery Book 93,Author 59,Mystery,0.789383
3,37,Fantasy Book 37,Author 7,Fantasy,0.768045
4,151,Science Fiction Book 151,Author 20,Science Fiction,0.758007
5,79,Fantasy Book 79,Author 49,Fantasy,0.757395
6,96,Science Fiction Book 96,Author 7,Science Fiction,0.753472
7,97,Science Fiction Book 97,Author 33,Science Fiction,0.752389
8,134,Horror Book 134,Author 24,Mystery,0.750338
9,166,Science Fiction Book 166,Author 13,Science Fiction,0.749133


In [ ]:
uid = 1
ratios = user_interest_ratios(uid, interactions_df)
feed = build_mixed_feed(uid, interactions_df, feed_size=50)

print("Interest ratios (top):")
print(dict(sorted(ratios.items(), key=lambda x: x[1], reverse=True)[:5]))

print("\nFeed genre counts:")
print(feed["genre"].value_counts())


Interest ratios (top):
{'Science Fiction': 0.2222222222222222, 'Mystery': 0.2037037037037037, 'Fantasy': 0.1111111111111111, 'Children': 0.1111111111111111, 'Poetry': 0.1111111111111111}

Feed genre counts:
genre
Science Fiction    11
Mystery            10
Fantasy             6
Children            6
Poetry              6
Romance             4
History             3
Self-Help           3
Business            3
Name: count, dtype: int64


In [ ]:
user_feed_state = {}  # user_id -> set(book_ids_shown)

def get_next_feed_page(user_id, interactions_df, page_size=20):
    shown = user_feed_state.get(user_id, set())

    # add temporary "view" interactions for already shown books
    if len(shown) > 0:
        temp_rows = pd.DataFrame({
            "user_id": [user_id]*len(shown),
            "book_id": list(shown),
            "event_type": ["view"]*len(shown),
            "timestamp": [datetime.now().isoformat()]*len(shown)
        })
        temp_interactions = pd.concat([interactions_df, temp_rows], ignore_index=True)
    else:
        temp_interactions = interactions_df

    page = build_mixed_feed(user_id, temp_interactions, feed_size=page_size, random_state=np.random.randint(0, 10_000))

    # update shown state
    user_feed_state[user_id] = shown.union(set(page["book_id"].astype(int).tolist()))
    return page

uid = 1
page1 = get_next_feed_page(uid, interactions_df, page_size=20)
page2 = get_next_feed_page(uid, interactions_df, page_size=20)

page1.head(10), page2.head(10)


(   book_id                     title     author            genre     score
 0      114          Mystery Book 114  Author 25          Mystery  0.809269
 1       93           Mystery Book 93  Author 59          Mystery  0.789383
 2       37           Fantasy Book 37   Author 7          Fantasy  0.768045
 3      164  Science Fiction Book 164  Author 47  Science Fiction  0.757111
 4      137  Science Fiction Book 137  Author 21  Science Fiction  0.754149
 5       26   Science Fiction Book 26   Author 6  Science Fiction  0.747521
 6       57   Science Fiction Book 57  Author 37  Science Fiction  0.747497
 7      189  Science Fiction Book 189  Author 52  Science Fiction  0.745348
 8       21            Poetry Book 21   Author 2           Poetry  0.742488
 9       15   Science Fiction Book 15  Author 57  Science Fiction  0.741855,
    book_id                     title     author            genre     score
 0      124          Mystery Book 124  Author 55          Mystery  0.793147
 1       14

In [ ]:
!pip -q install gradio


In [ ]:
import gradio as gr

def reset_user(uid: int):
    user_feed_state.pop(uid, None)
    return f"Reset done for user {uid}"

def show_ratios(uid: int):
    ratios = user_interest_ratios(uid, interactions_df)
    top = sorted(ratios.items(), key=lambda x: x[1], reverse=True)[:5]
    return "\n".join([f"{g}: {v:.2f}" for g, v in top])

def next_page(uid: int, page_size: int):
    page = get_next_feed_page(uid, interactions_df, page_size=page_size)
    return page

with gr.Blocks() as demo:
    gr.Markdown("# 📚 AI Book Recommendation Feed (Mixed)")

    with gr.Row():
        uid = gr.Number(value=1, label="User ID", precision=0)
        page_size = gr.Slider(5, 50, value=20, step=5, label="Page size")

    with gr.Row():
        btn_ratios = gr.Button("Show Interest Ratios")
        btn_next = gr.Button("Load Next Page")
        btn_reset = gr.Button("Reset Feed")

    ratios_out = gr.Textbox(label="Top Interest Ratios", lines=6)
    table_out = gr.Dataframe(label="Recommended Feed")

    btn_ratios.click(show_ratios, inputs=[uid], outputs=[ratios_out])
    btn_next.click(next_page, inputs=[uid, page_size], outputs=[table_out])
    btn_reset.click(reset_user, inputs=[uid], outputs=[ratios_out])

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://508c17e883337f6f8c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
